# Audit and metrics EDA

This notebook only loads artifacts. It does not train, parse raw txt, or read parquet.

Run from the **repo root** (kernel cwd = repository root). If a file is missing, the cell should fail; do not fill in numbers by hand.

- Audit CSVs: `data/audit/sessions.csv`, `coverage.csv`, `missing_cells.csv`, `hz_by_session.csv` (from `python -m har.data.audit`)
- Metrics JSON: `docs/reports/*.json` (from `python -m har.train` / `python -m har.evaluate`)


In [ ]:
from pathlib import Path
import json

import pandas as pd

AUDIT = Path("data/audit")
REPORTS = Path("docs/reports")

sessions = pd.read_csv(AUDIT / "sessions.csv")
coverage = pd.read_csv(AUDIT / "coverage.csv")
missing = pd.read_csv(AUDIT / "missing_cells.csv")
hz = pd.read_csv(AUDIT / "hz_by_session.csv")

sessions.head(), len(sessions), len(missing)

## Missing cells and implied Hz

Loaded from `data/audit/`. This dump has missing subject-activity-stream cells; a non-empty `missing_cells.csv` is expected. An empty table would mean the audit found no zero-sample cells.


In [ ]:
missing


In [ ]:
hz["implied_hz"].describe()


## Train / evaluate metrics JSON

Loads every `docs/reports/*.json` except `ladder_summary.json`. Missing files are skipped only when they do not exist on disk; this cell still fails if the reports directory is absent. Do not paste numbers that are not in those files.


In [ ]:
rows = []
for path in sorted(REPORTS.glob("*.json")):
    if path.name == "ladder_summary.json":
        continue
    payload = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(payload, dict) or "macro_f1" not in payload:
        continue
    rows.append(
        {
            "source": path.name,
            "protocol": payload.get("protocol"),
            "protocol_name": payload.get("protocol_name"),
            "device": payload.get("device"),
            "features": payload.get("features"),
            "model": payload.get("model"),
            "macro_f1": payload["macro_f1"],
            "accuracy": payload.get("accuracy"),
        }
    )

metrics = pd.DataFrame(rows)
metrics

## Explicit Protocol A2 load

`docs/reports/protocol_a_leaky.json` must exist. If it does not, this cell fails.


In [ ]:
a2 = json.loads((REPORTS / "protocol_a_leaky.json").read_text(encoding="utf-8"))
{
    "protocol": a2["protocol"],
    "protocol_name": a2["protocol_name"],
    "macro_f1": a2["macro_f1"],
    "accuracy": a2["accuracy"],
}
